In [ ]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import networkx as nx
from pathlib import Path
import matplotlib.pyplot as plt

sys.path.insert(0 , './../MAIN/')
from network_functions import (
    build_phenotype_KG,
    attach_phenotype_flags_from_df,
    normalize_edges_similarity_from_node_attr,
    fix_pheno_onehot_in_graph,
    knn_threshold
)

datadir = "/work/gr-fe/bryan/data/YEAST/"

netdir = Path(datadir) / "Networks"
netdir.mkdir(parents=True, exist_ok=True)

prefix = 'KGene'
out_train = netdir / f"{prefix}.gpickle"

# Load data
with open(f"{datadir}/02_processed/phenotype.processed.pkl", "rb") as file:
    data = pickle.load(file)

# Filter min phecode counts and patient counts (same logic as notebook)
min_count = 1
phecode_to_keep = data['Phenotype'].value_counts()[data['Phenotype'].value_counts() >= min_count].index
data = data[data['Phenotype'].isin(phecode_to_keep)]

# Build graph
print('Building KG')
G = build_phenotype_KG(data[['ID', 'Phenotype']] , id_col = 'ID' , phenotype_col = 'Phenotype')
#print(G.number_of_nodes(), G.number_of_edges())

# Normalize edges (set chosen metric as weight)
G = normalize_edges_similarity_from_node_attr(G, metric='cosine', out_attr='weight', replace_weight=True)

# Attach flags/attributes
G = attach_phenotype_flags_from_df(G, data)

# Fix node attribute in place
count = fix_pheno_onehot_in_graph(G)
print(f"Converted 'phenotype_onehot' to numpy arrays for {count} nodes.")

# Quick sanity check
try:
    any_type = next(iter(G.nodes(data=True)))[1].get("phenotype_onehot", None)
    print(f"Example 'phenotype_onehot' type: {type(any_type)}")
except StopIteration:
    print("Graph has no nodes to check.")

# Threshold edges 
G_train , knn = knn_threshold(G , k=20 , kind='similarity' , mode='union')

# Save results
print(f"Saving knowledge graph to {out_train}")
with open(out_train, 'wb') as f:
    pickle.dump(G_train, f, pickle.HIGHEST_PROTOCOL)

print("Done.")


Building KG
['0', '1', '10', '11', '12', '13', '2', '3', '4', '5', '6', '7', '8', '9']
Converted 'phenotype_onehot' to numpy arrays for 0 nodes.
Example 'phenotype_onehot' type: <class 'NoneType'>
